# سبق 10 - پیداوار میں AI ایجنٹس

اس سبق میں آپ Microsoft Agent Framework (Python) کے استعمال سے AI ایجنٹس کے لیے **پیداوار کے پیٹرنز** سیکھیں گے۔ ہم شامل ہیں:

- **مشاہدہ کاری** — ایجنٹ تعاملات میں وقت اور لاگنگ شامل کرنا
- **تشخیص** — جواب کی کوالٹی کو اسکور کرنے کے لیے ایک ایویلیوایٹر ایجنٹ کا استعمال
- **لاگت کا انتظام** — ٹوکن آپٹیمائزیشن اور ماڈل کے انتخاب کے لیے حکمت عملیاں

منظرنامہ ایک **ٹریول ایجنٹ** ہے جو صارفین کو سفر کی منصوبہ بندی میں مدد دیتا ہے، جس پر نگرانی اور تشخیص کی تہہ چڑھائی گئی ہے۔


## سیٹ اپ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## پیداواری غور و فکر

AI ایجنٹس کو پروٹوٹائپس سے پیداواری ماحول میں منتقل کرنا تین ستونوں پر گہری توجہ کا متقاضی ہے:

1. **مشاہداتی صلاحیت** — آپ کو یہ دیکھنے کی ضرورت ہے کہ ایجنٹ کیا کر رہا ہے، اسے کتنا وقت لگتا ہے، اور کون سے اوزار استعمال کیے جا رہے ہیں۔ بغیر ٹریسنگ اور لاگنگ کے، پیداواری مسائل کو ٹھیک کرنا تقریباً ناممکن ہے۔

2. **تشخیص** — خودکار معیار کے چیک یقینی بناتے ہیں کہ ایجنٹ کے جوابات وقت کے ساتھ درست، مکمل، اور مددگار رہیں۔ ایک ایویلیوایٹر ایجنٹ جوابات کو متعین معیار کے خلاف اسکور کر سکتا ہے۔

3. **لاگت کا انتظام** — ٹوکن کے استعمال سے لاگت براہ راست متاثر ہوتی ہے۔ پرامپٹ کی اصلاح، ماڈل کا انتخاب، اور کیشنگ جیسی حکمت عملیاں بغیر معیار سے سمجھوتہ کیے اخراجات کو قابو میں رکھنے میں مدد کرتی ہیں۔


## ایک قابل مشاہدہ ایجنٹ بنانا

ہم سفر کے اوزار کی تعریف کرتے ہیں اور تاخیر کی نگرانی کے لیے ایجنٹ کال کو وقت کے ساتھ لپیٹتے ہیں۔ پیداواری ماحول میں آپ OpenTelemetry یا اسی طرح کے ٹریسنگ بیک اینڈ کے ساتھ انضمام کریں گے۔


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## جائزہ کے نمونے

ایک عام پیداوار کا نمونہ یہ ہے کہ دوسرے ایجنٹ کو ایک **تجزیہ کار** کے طور پر استعمال کیا جائے۔ تجزیہ کار ابتدائی ایجنٹ کے جواب کو پہلے سے متعین کردہ معیارات جیسے کہ مکمل ہونے، درستگی، اور مدد فراہم کرنے کے لحاظ سے اسکور کرتا ہے۔

اس سے ممکن ہوتا ہے:
- صارفین تک پہنچنے والے جوابات سے پہلے خودکار معیار کی جانچ
- پرامپٹس یا ماڈلز میں تبدیلی پر واپسی کی شناخت
- وقت کے ساتھ ایجنٹ کی کارکردگی کی مسلسل نگرانی


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## لاگت کا انتظام کی حکمت عملیاں

لاگتوں کو کنٹرول کرنا پیداوار کے AI ایجنٹس کے لیے بہت اہم ہے۔ یہاں کلیدی حکمت عملیاں ہیں:

| حکمت عملی | وضاحت |
|---|---|
| **پرومپٹ کی اصلاح** | نظام کی ہدایات کو مختصر رکھیں۔ ان پٹ ٹوکنز کو کم کرنے کے لیے غیر ضروری سیاق و سباق ہٹا دیں۔ |
    "| **ماڈل کا انتخاب** | آسان کاموں جیسے درجہ بندی یا استخراج کے لیے چھوٹے، سستے ماڈلز (مثلاً GPT-5-mini) استعمال کریں، اور پیچیدہ استدلال کے لیے بڑے ماڈلز کو محفوظ رکھیں۔ |\n",
| **کیشنگ** | ٹول کے نتائج اور بار بار پوچھے جانے والے سوالات کو کیش کریں تاکہ غیر ضروری API کالز سے بچا جا سکے۔ |
| **ٹوکن بجٹ** | غیر متوقع طور پر لمبے جوابات کو روکنے کے لیے `max_tokens` کی حد مقرر کریں۔ |
| **بیچنگ** | جہاں ممکن ہو، متعدد صارف کے سوالات کو ایک API کال میں گروپ کریں۔ |

عملی طور پر، ایک مرحلہ وار طریقہ کار بہتر کام کرتا ہے: آسان درخواستوں کو تیز، سستے ماڈل پر بھیجیں اور صرف پیچیدہ سوالات کو زیادہ قابل ماڈل کی طرف بڑھائیں۔


## خلاصہ

اس سبق میں آپ نے سیکھا کہ کیسے:

1. **مشاہدہ پذیری شامل کریں** ایجنٹ تعاملات میں وقت اور لاگنگ کے ساتھ، تاکہ ٹریسنگ اور مانیٹرنگ کی بنیاد رکھی جا سکے۔
2. **ایجنٹ کے جوابات کا خودکار جائزہ لیں** ایک ایویلیوایٹر ایجنٹ کے ذریعے جو مکملت، درستگی، اور مددگی کو اسکور کرتا ہے۔
3. **اخراجات کا انتظام کریں** پرامپٹ آپٹیمائزیشن، ماڈل کا انتخاب، کیشنگ، اور ٹوکن بجٹ کے ذریعے۔

یہ پروڈکشن کے نمونے یقینی بناتے ہیں کہ آپ کے AI ایجنٹس قابلِ اعتماد، قابلِ ماپ، اور لاگت-موثر ہیں جب وہ وسیع پیمانے پر کام کر رہے ہوں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ڈس کلیمر**:
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ جبکہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات سے آگاہ رہیں کہ خودکار ترجمے میں غلطیاں یا عدم درستیاں ہو سکتی ہیں۔ اصل دستاویز اپنے مادری زبان میں مستند ماخذ سمجھی جائے گی۔ حساس معلومات کے لیے پیشہ ور انسانی ترجمہ کی سفارش کی جاتی ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والی کسی بھی غلط فہمی یا غلط تشریح کی ذمہ داری ہم قبول نہیں کرتے۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
